In [1]:
!pip install tqdm

import nltk as nltk
nltk.download('stopwords')
nltk.download('punkt')
nltk.download('punkt_tab')

from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk.util import ngrams
from tqdm import tqdm
from collections import defaultdict, Counter
import numpy as np
import math as math

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.
[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.


In [2]:
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


In [3]:
import pandas as pd
import numpy as np

stop_words = set(stopwords.words('english'))
df = pd.DataFrame(pd.read_json('/content/drive/MyDrive/Information_Retrieval/BM25/data/corpus.jsonl', lines=True))
df.drop(columns=['metadata'], inplace=True)
corpus_tokens = {}

def tokenize(text):
    tokens = word_tokenize(text.lower())
    filtered_tokens = [word for word in tokens if word.isalnum() and word not in stop_words]
    return filtered_tokens

for index, row in tqdm(df.iterrows(), total=df.shape[0]):
    tokens = tokenize(row['text'])
    filtered_tokens = [word for word in tokens if word.isalnum() and word not in stop_words]
    corpus_tokens[row['_id']] = filtered_tokens

100%|██████████| 171332/171332 [03:19<00:00, 858.77it/s]


In [4]:
inverted_index = defaultdict(dict)
for doc_id, tokens in tqdm(corpus_tokens.items(), desc='Indexing...'):
    for term, frequency in Counter(tokens).items():
        inverted_index[term][doc_id] = frequency

Indexing...: 100%|██████████| 171332/171332 [00:12<00:00, 14035.98it/s]


In [5]:
docs_len = {}
for index, row in tqdm(df.iterrows(), total=df.shape[0], desc='Calculating doc stats...'):
    docs_len[row['_id']] = len(corpus_tokens[row['_id']])

Calculating doc stats...: 100%|██████████| 171332/171332 [00:08<00:00, 19226.80it/s]


In [6]:
N = len(df)
average_dl = sum(docs_len.values()) / N

def bm25_score(term, doc_id, k1=0.50, b=0.75):
  if term not in inverted_index or doc_id not in inverted_index[term]:
    return 0.0

  tf = inverted_index[term][doc_id]
  dl = docs_len[doc_id]
  df = len(inverted_index[term])
  idf = math.log((N - df + 0.5) / (df + 0.5))
  denom = tf + k1 * (1 - b + b * dl / average_dl)
  score = idf * (tf * (k1 + 1) / denom)
  return score


In [7]:
query = 'what causes death from Covid-19'
query_tokens = tokenize(query)
union_docs = set().union(*(inverted_index[t].keys() for t in query_tokens))

scores = defaultdict(float)
for doc_id in tqdm(union_docs, desc='Calculating scores...'):
    score = sum(bm25_score(t, doc_id) for t in query_tokens)
    scores[doc_id] = score

sorted_scores = sorted(scores.items(), key=lambda x: x[1], reverse=True)
sorted_scores = sorted_scores[:50]
sorted_scores

Calculating scores...: 100%|██████████| 10641/10641 [00:00<00:00, 380581.80it/s]


[('9yb9a9vz', 8.91369926348563),
 ('464cqc16', 8.873009662621634),
 ('0q00yq1j', 8.80093135197588),
 ('0nyy7vjo', 8.699652563375798),
 ('f0drff3l', 8.62688606489817),
 ('sefu1ssn', 8.579395345907734),
 ('ql1pewdj', 8.563891854974083),
 ('z2jdiee1', 8.521973303541454),
 ('qqd3hxue', 8.50787963152803),
 ('gopgjsir', 8.505408994451528),
 ('ku5nkif6', 8.471614175491059),
 ('md9om47x', 8.292529475682903),
 ('eqbaw2wp', 8.257679917196297),
 ('iultl80s', 8.2271312807134),
 ('gtp01rna', 8.208697670265444),
 ('szf5n272', 8.191383006738228),
 ('am2njc2d', 8.178232478869212),
 ('p49vahoe', 8.16920052994274),
 ('4qqyb8sd', 8.144037307415138),
 ('aj8nxi3x', 8.127544851236937),
 ('lf90j7mm', 8.099445236379198),
 ('z68j0c63', 8.070588819321213),
 ('lizwiate', 8.06086816344029),
 ('76aeat80', 8.046903748237987),
 ('o9aat90h', 7.958983240101443),
 ('tm1s6wvj', 7.947230654981806),
 ('d8rlx6j3', 7.946151121840253),
 ('p2eh77es', 7.946081470329947),
 ('w9i57ubf', 7.85625723753445),
 ('3c1lo20c', 7.8281378